# 🛡️ SATARK — ML Fast Track: Model Training, Evaluation & ONNX Export
### High-Precision Trilingual Scam Detection (English, Roman Hinglish, Devanagari Hindi)

**Target Architectures:**
- **Primary Model:** `FacebookAI/xlm-roberta-base` (Trilingual Multilingual Transformer Baseline)
- **Secondary Candidate:** `l3cube-pune/hing-bert` (Roman Hinglish Specialist Comparison)

---

### 🏛️ Strict 4-Partition Isolation & Leakage Firewall (Total Corpus: 4,500 Rows)

To prevent data contamination, optimistic bias, and false claims of production readiness, this pipeline enforces strict boundaries across four distinct partitions:

| Partition | Size | Source | Role & Sovereign Boundary | Gradient / Tuning Exposure |
|:---|:---:|:---|:---|:---:|
| **Internal Training** (`train_internal`) | **2,835 rows** (63%) | Stratified 90% split of `train.csv` | Primary model parameter optimization (backpropagation). | **YES** (Epoch training) |
| **Internal Validation** (`val_internal`) | **315 rows** (7%) | Stratified 10% split of `train.csv` | Checkpoint selection (`load_best_model_at_end=True, metric_for_best_model="f1"`) and **pre-holdout model comparison**. | **NO** (Eval only) |
| **Calibration** (`calibration.csv`) | **675 rows** (15%) | Dedicated partition | **Post-training threshold optimization ($T^*$)** against false alarms on legitimate banking, OTP negations, and utility notices. | **NO** (Post-train only) |
| **Untouched Holdout** (`holdout.csv`) | **675 rows** (15%) | Strictly barred partition | **Unbiased final validation only.** Never used to select models, thresholds, architectures, or hyperparameters. | **NO** (Single read) |

### 🔒 Architectural Principles Enforced:
1. **Pre-Holdout Architectural Decision:** The comparison between `XLM-RoBERTa` and `Hing-BERT` is conducted strictly on `val_internal` (315 rows). The holdout set is never used to decide architecture.
2. **Dedicated Calibration:** Decision threshold $T^*$ is optimized on `calibration.csv` (675 rows) using a visible, configurable objective ($F_1 - \lambda \cdot \text{FPR}$).
3. **Architecture-Aware ONNX Graph:** Explicit input tensor contracts (`input_ids`, `attention_mask` for XLM-R; `+ token_type_ids` for Hing-BERT) with runtime graph inspection and strict numerical parity ($< 10^{-4}$).
4. **Reproducible Provenance Manifest:** Generates `manifest.json` recording model identifier, label mapping, threshold formula, dataset split counts, and SHA-256 checksums of all artifact files.
5. **Preserved Evaluators & Governance:** The frozen evaluation fixture `eval/test_set.csv` is preserved untouched. Rule Engine remains sovereign, and `ML_ENABLED=false` remains standard.

---

### ⚡ Two Operating Modes Supported:
- **Mode A — Complete Training & Evaluation Pipeline:** Run Sections 1 through 14 sequentially to split datasets, fine-tune models, perform pre-holdout selection, calibrate threshold, evaluate on holdout, export to ONNX, and benchmark.
- **Mode B — Dedicated Standalone Export-Only Mode:** If you already have the fine-tuned and frozen XLM-RoBERTa checkpoint from Colab (or uploaded to `ml_service/artifacts/checkpoint`):
  1. Run **Section 1 & 2 (Environment Setup & Dependencies)**.
  2. Jump directly to **Section 11: Standalone Export-Only Workflow (Frozen Checkpoint -> ONNX)**!
  3. This exports the frozen checkpoint to ONNX opset 17, validates graph inputs, runs 10-sample parity validation at calibrated threshold $T^* = 0.10$ ($< 10^{-4}$ discrepancy), writes `model_metadata.json` and `manifest.json`, and packages `satark_xlm_roberta_export_bundle.zip` without invoking `Trainer.train()`, optimizer steps, calibration, or holdout evaluation.

## 1. Environment Setup & Dependency Installation
Install production NLP, PyTorch, Hugging Face, ONNX, and ONNX Runtime libraries.
**Migration status: UNVALIDATED.** The pinned Transformers 5 / PyTorch environment below requires Python 3.11+. Training, export, tokenizer parity, and Colab GPU compatibility must be rerun in an isolated environment; existing results and artifact manifests do not validate these new versions.
*Hardware Recommendation: Google Colab T4 GPU (free tier).*

In [ ]:
# Reviewed dependency pins; this migration has not been executed or validated.
import sys
assert sys.version_info >= (3, 11), "Transformers 5.22.1 requires Python 3.11+"
!pip install -q transformers==5.22.1 tokenizers==0.23.1 sentencepiece==0.2.3 datasets==5.1.0 accelerate==1.15.0 torch==2.15.0 onnx==1.23.2 onnxruntime==1.20.1 numpy==2.2.1 scikit-learn==1.6.0 tabulate==0.9.0

import os
import sys
import json
import time
import shutil
import random
import hashlib
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Set deterministic random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("SATARK ML TRAINING RUNTIME ENVIRONMENT")
print("=" * 70)
print(f"Python Version:    {sys.version.split()[0]}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:    {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("WARNING: Running on CPU. Training will take significantly longer.")
print(f"Deterministic Seed: {SEED}")
print("=" * 70)

## 2. Dataset Ingestion & Storage Setup
Ensure the pre-split SATARK corpus files are available in `ml_service/dataset/`:
- `train.csv` (3,150 rows - 70% of total corpus)
- `calibration.csv` (675 rows - 15% of total corpus)
- `holdout.csv` (675 rows - 15% of total corpus)

In [ ]:
# Setup workspace directory structure
DATA_DIR = Path("ml_service/dataset")
ARTIFACTS_DIR = Path("ml_service/artifacts")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# If running in Colab without mounted files, trigger direct file upload
train_file = DATA_DIR / "train.csv"
calib_file = DATA_DIR / "calibration.csv"
holdout_file = DATA_DIR / "holdout.csv"

if not (train_file.exists() and calib_file.exists() and holdout_file.exists()):
    try:
        from google.colab import files
        print("Please upload train.csv, calibration.csv, and holdout.csv:")
        uploaded = files.upload()
        for filename, content in uploaded.items():
            dest = DATA_DIR / filename
            with open(dest, "wb") as f:
                f.write(content)
            print(f"Saved {filename} -> {dest}")
    except ImportError:
        print("Local runtime detected. Verifying files at:", DATA_DIR)

for fpath in [train_file, calib_file, holdout_file]:
    assert fpath.exists(), f"Missing required dataset file: {fpath}"
print("All 3 primary dataset files located successfully.")

## 3. Strengthened Dataset Audit & Leakage Verification
We perform comprehensive static checks on all three partitions:
1. **Required Columns:** `id`, `text`, `label`, `language`, `category` (and governance columns).
2. **Non-Empty Text:** Zero missing, null, or empty whitespace strings.
3. **Canonical Values:** Canonical labels (`BENIGN`, `SUSPICIOUS`) and valid languages (`en`, `hinglish`, `hi`).
4. **Explicit Row Counts:** Exactly 3,150 train, 675 calibration, 675 holdout (4,500 total).
5. **Class Balance Assertions:** Perfect 1.0 balance in training (1,575 / 1,575) and balanced calibration/holdout.
6. **Collision & Leakage Checks:** Zero exact duplicates, zero normalized-text collisions, and zero template fingerprint leakage across all splits.

In [ ]:
import re

def extract_template_fingerprint(text: str) -> str:
    """
    Extracts a generalized structural template fingerprint from message text
    by normalizing specific entities (numbers, amounts, URLs, phones, dates).
    Used to guarantee zero template-family leakage across splits.
    """
    if not text or not isinstance(text, str):
        return ""
    lowered = text.lower().strip()
    # 1. Normalize URLs and domains
    t = re.sub(
        r'(?:hxxps?|https?|ftp)://\S+|www\.\S+|\b[a-zA-Z0-9.-]+\.(?:com|in|org|net|xyz|top|site|online|live|click|app|cc|co)\b(?:/\S*)?',
        '<url>',
        lowered,
        flags=re.IGNORECASE
    )
    # 2. Normalize Indian phone numbers and masked phone sequences
    t = re.sub(r'\b(?:\+?91[\-\s]?)?[6-9]\d{9}\b|\b\d{2}[xX]{5,8}\d{2}\b', '<phone>', t)
    # 3. Normalize currency markers and amounts
    t = re.sub(r'(?:rs\.?|inr|₹)\s*[\d,]+(?:\.\d+)?', '<amt>', t)
    # 4. Normalize standalone dates / times
    t = re.sub(r'\b\d{1,2}[-/](?:[a-z]{3}|\d{1,2})[-/]\d{2,4}\b|\b\d{1,2}:\d{2}(?:\s*(?:am|pm))?\b', '<time>', t)
    # 5. Normalize account masking patterns (e.g. XX1049, A/c ...4912)
    t = re.sub(r'[xX]{2,}\d{2,}', '<acct>', t)
    # 6. Normalize remaining standalone digits
    t = re.sub(r'\b\d+\b', '<num>', t)
    # 7. Collapse punctuation and whitespace
    t = re.sub(r'[^\w\s<>]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def audit_dataset_integrity(data_dir: Path):
    REQUIRED_COLS = ["id", "text", "label", "language", "category"]
    VALID_LABELS = {"BENIGN", "SUSPICIOUS"}
    VALID_LANGS = {"en", "hinglish", "hi"}

    df_train = pd.read_csv(data_dir / "train.csv")
    df_calib = pd.read_csv(data_dir / "calibration.csv")
    df_holdout = pd.read_csv(data_dir / "holdout.csv")

    print("=" * 75)
    print("SATARK COMPREHENSIVE DATASET INTEGRITY & LEAKAGE AUDIT")
    print("=" * 75)

    # 1. Row Count Verification
    assert len(df_train) == 3150, f"Train count mismatch: expected 3150, got {len(df_train)}"
    assert len(df_calib) == 675, f"Calibration count mismatch: expected 675, got {len(df_calib)}"
    assert len(df_holdout) == 675, f"Holdout count mismatch: expected 675, got {len(df_holdout)}"
    total_rows = len(df_train) + len(df_calib) + len(df_holdout)
    assert total_rows == 4500, f"Total corpus mismatch: expected 4500, got {total_rows}"
    print(f"Row Counts:           PASSED (Train: 3,150 | Calib: 675 | Holdout: 675 | Total: 4,500)")

    # 2. Schema and Null Checks
    for name, df in [("Train", df_train), ("Calibration", df_calib), ("Holdout", df_holdout)]:
        missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
        assert not missing_cols, f"{name} set missing columns: {missing_cols}"
        assert df["text"].notnull().all(), f"{name} set has null texts"
        assert (df["text"].str.strip().str.len() > 0).all(), f"{name} set has empty texts"
        assert set(df["label"].unique()).issubset(VALID_LABELS), f"{name} set has invalid labels"
        assert set(df["language"].unique()).issubset(VALID_LANGS), f"{name} set has invalid languages"

    print("Schema & Field Validity: PASSED (Required columns present, no nulls/empty strings, valid canonical enums)")

    # 3. Class Balance Verification
    train_benign = (df_train["label"] == "BENIGN").sum()
    train_scam = (df_train["label"] == "SUSPICIOUS").sum()
    assert train_benign == 1575 and train_scam == 1575, f"Train imbalance: {train_benign} BENIGN vs {train_scam} SUSPICIOUS"
    calib_bal = (df_calib["label"] == "SUSPICIOUS").mean()
    holdout_bal = (df_holdout["label"] == "SUSPICIOUS").mean()
    assert 0.45 <= calib_bal <= 0.55, f"Calib imbalance: {calib_bal:.3f}"
    assert 0.45 <= holdout_bal <= 0.55, f"Holdout imbalance: {holdout_bal:.3f}"
    print(f"Class Balance:        PASSED (Train 1:1 [1575/1575] | Calib [{calib_bal:.3f}] | Holdout [{holdout_bal:.3f}])")

    # 4. Intra-Split Duplicate Checks
    for name, df in [("Train", df_train), ("Calibration", df_calib), ("Holdout", df_holdout)]:
        exact_dups = df["text"].duplicated().sum()
        norm_dups = df["text"].str.strip().str.lower().duplicated().sum()
        assert exact_dups == 0, f"{name} set has {exact_dups} exact duplicates"
        assert norm_dups == 0, f"{name} set has {norm_dups} normalized text duplicates"
    print("Intra-Split Collisions: PASSED (0 exact duplicates, 0 normalized collisions within partitions)")

    # 5. Cross-Split Normalized Text Collisions
    norm_train = set(df_train["text"].str.strip().str.lower())
    norm_calib = set(df_calib["text"].str.strip().str.lower())
    norm_holdout = set(df_holdout["text"].str.strip().str.lower())

    tc_overlap = norm_train.intersection(norm_calib)
    th_overlap = norm_train.intersection(norm_holdout)
    ch_overlap = norm_calib.intersection(norm_holdout)

    assert len(tc_overlap) == 0, f"DATA LEAKAGE: {len(tc_overlap)} text matches between Train and Calibration!"
    assert len(th_overlap) == 0, f"DATA LEAKAGE: {len(th_overlap)} text matches between Train and Holdout!"
    assert len(ch_overlap) == 0, f"DATA LEAKAGE: {len(ch_overlap)} text matches between Calibration and Holdout!"
    print("Cross-Split Collisions: PASSED (0 text matches across Train, Calibration, and Holdout)")

    # 6. Cross-Split Template-Family Leakage
    fp_train = set(df_train["text"].apply(extract_template_fingerprint))
    fp_calib = set(df_calib["text"].apply(extract_template_fingerprint))
    fp_holdout = set(df_holdout["text"].apply(extract_template_fingerprint))

    assert len(fp_train.intersection(fp_calib)) == 0, "Template fingerprint leakage between Train and Calib!"
    assert len(fp_train.intersection(fp_holdout)) == 0, "Template fingerprint leakage between Train and Holdout!"
    assert len(fp_calib.intersection(fp_holdout)) == 0, "Template fingerprint leakage between Calib and Holdout!"
    print("Template Leakage Check: PASSED (0 structural template fingerprint overlaps across splits)")
    print("=" * 75)

    return df_train, df_calib, df_holdout

df_train, df_calib, df_holdout = audit_dataset_integrity(DATA_DIR)

## 4. Internal Training / Validation Split (2,835 Train / 315 Val)
We split `train.csv` (3,150 rows) into:
- **`df_train_internal`:** 2,835 rows (90%) for gradient descent fine-tuning.
- **`df_val_internal`:** 315 rows (10%) for checkpoint selection and early model comparison.

**Split Governance Rules:**
1. Grouping by template fingerprint guarantees **zero template leakage** between internal training and internal validation.
2. Stratification across all 6 `(label, language)` strata preserves exact demographic distributions.
3. **Calibration (`calibration.csv`) and Holdout (`holdout.csv`) are strictly untouched and never used in this split.**

In [ ]:
from collections import defaultdict

def create_internal_train_val_split(df_train: pd.DataFrame, seed: int = 42):
    """
    Splits the 3,150-row training partition into:
      - 2,835 internal training rows (90%)
      - 315 internal validation rows (10%)
    Preserves template-family grouping and stratifies across (label, language).
    Does NOT touch calibration or holdout.
    """
    rng = random.Random(seed)
    df = df_train.copy()
    df["fp"] = df["text"].apply(extract_template_fingerprint)
    df["stratum"] = df["label"].astype(str) + "__" + df["language"].astype(str)

    # Group row indices by (stratum, template_fingerprint)
    strata_clusters = defaultdict(list)
    fp_to_indices = defaultdict(list)
    for idx, row in df.iterrows():
        fp_to_indices[row["fp"]].append(idx)

    for fp, indices in fp_to_indices.items():
        st = df.loc[indices[0], "stratum"]
        strata_clusters[st].append(indices)

    # Exact target validation count per stratum (55*4 + 47 + 48 = 315)
    target_val_counts = {
        "SUSPICIOUS__en": 55,
        "SUSPICIOUS__hinglish": 55,
        "BENIGN__en": 55,
        "BENIGN__hinglish": 55,
        "BENIGN__hi": 47,
        "SUSPICIOUS__hi": 48
    }

    val_indices = []
    train_indices = []

    for st, clusters in sorted(strata_clusters.items()):
        rng.shuffle(clusters)
        target = target_val_counts[st]
        curr = 0
        st_val = []
        st_train = []
        for c in clusters:
            if curr + len(c) <= target:
                st_val.extend(c)
                curr += len(c)
            else:
                st_train.extend(c)
        val_indices.extend(st_val)
        train_indices.extend(st_train)

    df_train_internal = df.loc[train_indices].drop(columns=["fp", "stratum"]).reset_index(drop=True)
    df_val_internal = df.loc[val_indices].drop(columns=["fp", "stratum"]).reset_index(drop=True)

    # Rigorous Assertions
    assert len(df_train_internal) == 2835, f"Expected 2835 train rows, got {len(df_train_internal)}"
    assert len(df_val_internal) == 315, f"Expected 315 val rows, got {len(df_val_internal)}"
    assert len(df_train_internal) + len(df_val_internal) == 3150, "Row loss during internal split!"

    # Zero text or fingerprint leakage between train_internal and val_internal
    train_norm = set(df_train_internal["text"].str.strip().str.lower())
    val_norm = set(df_val_internal["text"].str.strip().str.lower())
    assert len(train_norm.intersection(val_norm)) == 0, "Text collision between train_internal and val_internal!"

    train_fps = set(df_train_internal["text"].apply(extract_template_fingerprint))
    val_fps = set(df_val_internal["text"].apply(extract_template_fingerprint))
    assert len(train_fps.intersection(val_fps)) == 0, "Template leakage between train_internal and val_internal!"

    print("=" * 75)
    print("INTERNAL TRAINING / VALIDATION SPLIT VERIFICATION")
    print("=" * 75)
    print(f"Internal Training Set:   {len(df_train_internal):>5} rows (BENIGN: {(df_train_internal['label']=='BENIGN').sum()}, SUSPICIOUS: {(df_train_internal['label']=='SUSPICIOUS').sum()})")
    print(f"Internal Validation Set: {len(df_val_internal):>5} rows (BENIGN: {(df_val_internal['label']=='BENIGN').sum()}, SUSPICIOUS: {(df_val_internal['label']=='SUSPICIOUS').sum()})")
    print(f"Template Leakage:        0 clusters shared between train_internal and val_internal")
    print(f"Calibration Partition:   675 rows (STRICTLY UNTOUCHED - RESERVED FOR THRESHOLD TUNING)")
    print(f"Holdout Partition:       675 rows (STRICTLY UNTOUCHED - RESERVED FOR FINAL EVALUATION)")
    print("=" * 75)

    return df_train_internal, df_val_internal

df_train_internal, df_val_internal = create_internal_train_val_split(df_train, seed=SEED)

## 5. Training Pipeline Definition
Define the Hugging Face Trainer pipeline:
1. Canonical label mapping (`BENIGN` -> 0, `SUSPICIOUS` -> 1).
2. Checkpoint selection based strictly on **`val_internal`** (`load_best_model_at_end=True, metric_for_best_model="f1"`).
3. **No calibration or holdout data is used during model training epochs or checkpoint selection.**

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

LABEL2ID = {"BENIGN": 0, "SUSPICIOUS": 1}
ID2LABEL = {0: "BENIGN", 1: "SUSPICIOUS"}

def normalize_label(label_str: str) -> str:
    s = str(label_str).strip().upper()
    if s in ["BENIGN", "0", "HAM"]:
        return "BENIGN"
    elif s in ["SUSPICIOUS", "1", "SCAM", "PHISHING", "FRAUD"]:
        return "SUSPICIOUS"
    raise ValueError(f"Unknown label: {label_str}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(labels, preds)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0
    return {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "fpr": round(fpr, 4),
        "fnr": round(fnr, 4)
    }

def train_satark_model(
    model_name: str,
    output_dir: Path,
    df_train_int: pd.DataFrame,
    df_val_int: pd.DataFrame,
    epochs: int = 3,
    batch_size: int = 16,
    lr: float = 2e-5,
    max_length: int = 128
):
    print(f"\n>>> Starting Fine-Tuning for: {model_name} <<<")
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        trust_remote_code=False,
        use_safetensors=True
    )

    def tokenize_function(batch):
        enc = tokenizer(batch["text"], truncation=True, max_length=max_length, padding="max_length")
        enc["labels"] = [LABEL2ID[normalize_label(l)] for l in batch["label"]]
        return enc

    train_ds = Dataset.from_pandas(df_train_int).map(tokenize_function, batched=True)
    val_ds = Dataset.from_pandas(df_val_int).map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir / "checkpoints"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Evaluate best checkpoint on internal validation set
    val_metrics = trainer.evaluate(val_ds)
    print(f"Best Checkpoint Internal Validation Results: F1 = {val_metrics.get('eval_f1', 0.0):.4f}, FPR = {val_metrics.get('eval_fpr', 0.0):.4f}")

    # Save best model and tokenizer
    final_model_dir = output_dir / "final_model"
    trainer.save_model(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))

    return final_model_dir, val_metrics, trainer

## 6. Train Primary Model: `FacebookAI/xlm-roberta-base`
Fine-tune the multilingual foundation model on `df_train_internal` (2,835 rows) with epoch checkpoint selection on `df_val_internal` (315 rows).

In [ ]:
XLM_OUTPUT_DIR = ARTIFACTS_DIR / "xlm_roberta"
xlm_model_dir, xlm_val_metrics, xlm_trainer = train_satark_model(
    model_name="FacebookAI/xlm-roberta-base",
    output_dir=XLM_OUTPUT_DIR,
    df_train_int=df_train_internal,
    df_val_int=df_val_internal,
    epochs=3,
    batch_size=16,
    lr=2e-5,
    max_length=128
)

## 7. Train Secondary Comparison Candidate: `l3cube-pune/hing-bert`
Fine-tune the Roman Hinglish specialist model on the same `df_train_internal` (2,835 rows), selecting the best checkpoint on the same `df_val_internal` (315 rows).

In [ ]:
HINGBERT_OUTPUT_DIR = ARTIFACTS_DIR / "hing_bert"
hingbert_model_dir, hingbert_val_metrics, hingbert_trainer = train_satark_model(
    model_name="l3cube-pune/hing-bert",
    output_dir=HINGBERT_OUTPUT_DIR,
    df_train_int=df_train_internal,
    df_val_int=df_val_internal,
    epochs=3,
    batch_size=16,
    lr=2e-5,
    max_length=128
)

## 8. Pre-Holdout Model Selection (Validation Evidence Only)
Evaluate both models on **`df_val_internal`** (315 rows) to make the architectural deployment decision.

> [!IMPORTANT]
> **Holdout Barrier Principle:** The holdout set (`holdout.csv`) remains strictly untouched. The architectural selection between `XLM-RoBERTa` and `Hing-BERT` is made exclusively using pre-holdout validation evidence.

In [ ]:
from tabulate import tabulate

def evaluate_on_validation_subgroups(model_dir: Path, df_val: pd.DataFrame):
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False, use_safetensors=True)
    model.eval()

    y_true, y_pred, y_probs = [], [], []
    latencies_ms = []

    for r in df_val.to_dict(orient="records"):
        text = str(r["text"])
        true_id = LABEL2ID[normalize_label(r["label"])]
        enc = tokenizer(text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        latencies_ms.append((time.perf_counter() - t0) * 1000.0)

        logits = out.logits.numpy()
        exp_l = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        prob = float((exp_l / np.sum(exp_l, axis=-1, keepdims=True))[0, 1])
        pred_id = 1 if prob >= 0.50 else 0

        y_true.append(true_id)
        y_pred.append(pred_id)
        y_probs.append(prob)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0

    # Subgroups
    languages = df_val["language"].str.strip().str.lower().tolist()
    per_lang = {}
    for lang in ["en", "hinglish", "hi"]:
        mask = [l == lang for l in languages]
        if sum(mask) > 0:
            p, r, f, _ = precision_recall_fscore_support(y_true[mask], y_pred[mask], average="binary", pos_label=1, zero_division=0)
            cm_sub = confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1])
            tn_s, fp_s = cm_sub[0, 0], cm_sub[0, 1] if cm_sub.size == 4 else (0, 0)
            per_lang[lang] = {
                "f1": round(f, 4),
                "precision": round(p, 4),
                "recall": round(r, 4),
                "fpr": round(float(fp_s / (fp_s + tn_s)) if (fp_s + tn_s) > 0 else 0.0, 4),
                "support": int(sum(mask))
            }

    return {
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1": round(f1, 4),
        "fpr": round(fpr, 4),
        "fnr": round(fnr, 4),
        "per_language": per_lang,
        "mean_latency_ms": round(float(np.mean(latencies_ms)), 2)
    }

print("Evaluating XLM-RoBERTa on Internal Validation Set (315 rows)...")
xlm_val_eval = evaluate_on_validation_subgroups(xlm_model_dir, df_val_internal)

print("Evaluating Hing-BERT on Internal Validation Set (315 rows)...")
hing_val_eval = evaluate_on_validation_subgroups(hingbert_model_dir, df_val_internal)

val_comparison = [
    ["Metric", "XLM-RoBERTa (Primary)", "Hing-BERT (Secondary)", "Delta (Hing - XLM)"],
    ["Val Overall Accuracy", f"{xlm_val_eval['accuracy']:.4f}", f"{hing_val_eval['accuracy']:.4f}", f"{hing_val_eval['accuracy']-xlm_val_eval['accuracy']:+.4f}"],
    ["Val Overall F1", f"{xlm_val_eval['f1']:.4f}", f"{hing_val_eval['f1']:.4f}", f"{hing_val_eval['f1']-xlm_val_eval['f1']:+.4f}"],
    ["Val Overall FPR", f"{xlm_val_eval['fpr']:.4f}", f"{hing_val_eval['fpr']:.4f}", f"{hing_val_eval['fpr']-xlm_val_eval['fpr']:+.4f}"],
    ["Val Hinglish F1", f"{xlm_val_eval['per_language']['hinglish']['f1']:.4f}", f"{hing_val_eval['per_language']['hinglish']['f1']:.4f}", f"{hing_val_eval['per_language']['hinglish']['f1']-xlm_val_eval['per_language']['hinglish']['f1']:+.4f}"],
    ["Val Hinglish FPR", f"{xlm_val_eval['per_language']['hinglish']['fpr']:.4f}", f"{hing_val_eval['per_language']['hinglish']['fpr']:.4f}", f"{hing_val_eval['per_language']['hinglish']['fpr']-xlm_val_eval['per_language']['hinglish']['fpr']:+.4f}"],
    ["Val Devanagari Hindi F1", f"{xlm_val_eval['per_language']['hi']['f1']:.4f}", f"{hing_val_eval['per_language']['hi']['f1']:.4f}", f"{hing_val_eval['per_language']['hi']['f1']-xlm_val_eval['per_language']['hi']['f1']:+.4f}"],
    ["Val English F1", f"{xlm_val_eval['per_language']['en']['f1']:.4f}", f"{hing_val_eval['per_language']['en']['f1']:.4f}", f"{hing_val_eval['per_language']['en']['f1']-xlm_val_eval['per_language']['en']['f1']:+.4f}"],
    ["CPU Mean Latency (ms)", f"{xlm_val_eval['mean_latency_ms']:.1f}", f"{hing_val_eval['mean_latency_ms']:.1f}", f"{hing_val_eval['mean_latency_ms']-xlm_val_eval['mean_latency_ms']:+.1f}"],
]

print(tabulate(val_comparison, headers="firstrow", tablefmt="fancy_grid"))

# PRE-HOLDOUT ARCHITECTURAL SELECTION DECISION
hinglish_delta = hing_val_eval["per_language"]["hinglish"]["f1"] - xlm_val_eval["per_language"]["hinglish"]["f1"]
if hinglish_delta > 0.02 and hing_val_eval["fpr"] <= xlm_val_eval["fpr"]:
    decision = "HYBRID ROUTING (Deploy Hing-BERT for Hinglish, XLM-RoBERTa for English/Hindi)"
    reason = f"Hing-BERT delivers a substantial +{hinglish_delta*100:.2f}% F1 gain on Roman Hinglish without increasing false alarm rate."
    selected_model_name = "l3cube-pune/hing-bert"
    selected_model_dir = hingbert_model_dir
else:
    decision = "SINGLE PRIMARY MODEL: FacebookAI/xlm-roberta-base"
    reason = (
        f"XLM-RoBERTa provides strong unified trilingual performance (Val F1: {xlm_val_eval['f1']:.4f}, Hinglish F1: {xlm_val_eval['per_language']['hinglish']['f1']:.4f}). "
        f"Hing-BERT does not show sufficient margin to justify doubling memory consumption, dual-model runtime routing, and operational overhead."
    )
    selected_model_name = "FacebookAI/xlm-roberta-base"
    selected_model_dir = xlm_model_dir

print("\n" + "=" * 75)
print(f"PRE-HOLDOUT SELECTION VERDICT: {decision}")
print(f"RATIONALE:                    {reason}")
print("GOVERNANCE NOTE:              Made strictly on validation evidence. Holdout data is UNTOUCHED.")
print("=" * 75)

with open(ARTIFACTS_DIR / "pre_holdout_model_selection.json", "w", encoding="utf-8") as f:
    json.dump({
        "selection_evidence": "val_internal (315 rows)",
        "decision": decision,
        "reason": reason,
        "selected_model": selected_model_name,
        "xlm_roberta_val": xlm_val_eval,
        "hing_bert_val": hing_val_eval
    }, f, indent=2)

## 9. Decision Threshold Optimization ($T^*$) on Calibration Partition
We optimize the decision threshold $T^*$ for the selected model using the dedicated **`calibration.csv`** (675 rows).

### Configurable Threshold Objective Function:
$$\text{Score}(T) = F_1(T) - (\lambda_{\text{FPR}} \cdot \text{FPR}(T))$$

Where $\lambda_{\text{FPR}} = 2.0$ (configurable) penalizes false alarms on legitimate banking, OTP warnings, and transactional notices.

> [!NOTE]
> The threshold $T^*$ is an operating point tailored for the specific risk penalty. In production, operators must monitor false alarm rates and adjust threshold according to business requirements.

In [ ]:
def optimize_threshold(model_dir: Path, df_calib: pd.DataFrame, lambda_fpr: float = 2.0):
    """
    Sweeps decision threshold on the dedicated 675-row calibration partition.
    Never uses holdout data.
    """
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False, use_safetensors=True)
    model.eval()

    y_true = []
    y_probs = []

    for r in df_calib.to_dict(orient="records"):
        text = str(r["text"])
        true_id = LABEL2ID[normalize_label(r["label"])]
        enc = tokenizer(text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

        with torch.no_grad():
            out = model(**enc)
        logits = out.logits.numpy()
        exp_l = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        prob = float((exp_l / np.sum(exp_l, axis=-1, keepdims=True))[0, 1])

        y_true.append(true_id)
        y_probs.append(prob)

    y_true = np.array(y_true)
    y_probs = np.array(y_probs)

    threshold_table = []
    best_score = -999.0
    best_t = 0.50
    best_row = {}

    for t in np.linspace(0.10, 0.90, 81):
        preds = (y_probs >= t).astype(int)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds, average="binary", pos_label=1, zero_division=0)
        cm = confusion_matrix(y_true, preds, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
        fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0

        # Objective: balance F1 while penalizing false positives
        score = f1 - (lambda_fpr * fpr)

        entry = {
            "threshold": round(float(t), 2),
            "precision": round(float(prec), 4),
            "recall": round(float(rec), 4),
            "f1": round(float(f1), 4),
            "fpr": round(float(fpr), 4),
            "fnr": round(float(fnr), 4),
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
            "score": round(float(score), 4)
        }

        if score > best_score:
            best_score = score
            best_t = float(t)
            best_row = entry

        # Sample for display table
        if round(float(t), 2) in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
            threshold_table.append([entry["threshold"], entry["precision"], entry["recall"], entry["f1"], entry["fpr"], entry["fnr"], entry["fp"], entry["score"]])

    # Insert optimal row
    threshold_table.append([f"{best_row['threshold']} (T*)", best_row['precision'], best_row['recall'], best_row['f1'], best_row['fpr'], best_row['fnr'], best_row['fp'], best_row['score']])
    threshold_table.sort(key=lambda x: float(str(x[0]).split()[0]))

    headers = ["Threshold (T)", "Precision", "Recall", "F1", "FPR", "FNR", "False Positives", "Score"]
    print("=" * 75)
    print("CALIBRATION THRESHOLD SWEEP TABLE (calibration.csv, 675 rows)")
    print(f"Objective Formula: Score = F1 - ({lambda_fpr} * FPR)")
    print("=" * 75)
    print(tabulate(threshold_table, headers=headers, tablefmt="fancy_grid"))

    print(f"\nOptimal Calibration Threshold Selected: T* = {best_t:.2f}")
    print(f"At T* = {best_t:.2f}: Precision={best_row['precision']}, Recall={best_row['recall']}, F1={best_row['f1']}, FPR={best_row['fpr']}")
    print("OPERATING DISCLAIMER: In production, operators must monitor live false alarm feedback and configure this threshold according to business risk tolerance.")
    print("=" * 75)

    return best_t, best_row

calib_threshold, calib_metrics = optimize_threshold(selected_model_dir, df_calib, lambda_fpr=2.0)

## 10. Final Untouched Holdout Evaluation
Now, for the **first time**, we evaluate the selected model (`FacebookAI/xlm-roberta-base`) at calibrated threshold $T^*$ on the **untouched holdout partition** (`holdout.csv`, 675 rows).

Reports:
1. Overall Accuracy, Precision, Recall, F1, FPR, FNR, Confusion Matrix.
2. Per-Language Metrics (`en`, `hinglish`, `hi`).
3. Category-Wise Metrics across all 12 canonical categories.
4. Hard-Negative False Positive Rate on legitimate banking, OTP negations, and utility notices.
5. CPU Inference Latency (mean, P50, P95).

In [ ]:
def evaluate_on_untouched_holdout(model_dir: Path, df_holdout: pd.DataFrame, threshold: float):
    """
    Single unbiased evaluation run on the untouched holdout partition.
    No model, architecture, or hyperparameter decisions are made here.
    """
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), local_files_only=True, trust_remote_code=False, use_safetensors=True)
    model.eval()

    y_true, y_pred, y_probs = [], [], []
    latencies_ms = []

    for r in df_holdout.to_dict(orient="records"):
        text = str(r["text"])
        true_id = LABEL2ID[normalize_label(r["label"])]
        enc = tokenizer(text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        latencies_ms.append((time.perf_counter() - t0) * 1000.0)

        logits = out.logits.numpy()
        exp_l = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        prob = float((exp_l / np.sum(exp_l, axis=-1, keepdims=True))[0, 1])
        pred_id = 1 if prob >= threshold else 0

        y_true.append(true_id)
        y_pred.append(pred_id)
        y_probs.append(prob)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0

    # Language Breakdown
    languages = df_holdout["language"].str.strip().str.lower().tolist()
    per_lang = {}
    for lang in ["en", "hinglish", "hi"]:
        mask = [l == lang for l in languages]
        if sum(mask) > 0:
            p, r, f, _ = precision_recall_fscore_support(y_true[mask], y_pred[mask], average="binary", pos_label=1, zero_division=0)
            cm_sub = confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1])
            tn_s, fp_s = cm_sub[0, 0], cm_sub[0, 1] if cm_sub.size == 4 else (0, 0)
            per_lang[lang] = {
                "f1": round(f, 4),
                "precision": round(p, 4),
                "recall": round(r, 4),
                "fpr": round(float(fp_s / (fp_s + tn_s)) if (fp_s + tn_s) > 0 else 0.0, 4),
                "support": int(sum(mask))
            }

    # Category Breakdown
    categories = df_holdout["category"].str.strip().str.lower().tolist()
    per_cat = {}
    for cat in sorted(set(categories)):
        mask = [c == cat for c in categories]
        if sum(mask) > 0:
            yt_c = y_true[mask]
            yp_c = y_pred[mask]
            correct = int(np.sum(yt_c == yp_c))
            per_cat[cat] = {
                "accuracy": round(correct / len(yt_c), 4),
                "support": len(yt_c),
                "false_alarms": int(np.sum((yt_c == 0) & (yp_c == 1))),
                "missed_scams": int(np.sum((yt_c == 1) & (yp_c == 0)))
            }

    # Hard-Negative False Positive Rate (Legitimate transactional notices)
    hn_cats = ["otp_negation", "bank_transactional", "utility_service_notice"]
    hn_fp, hn_total = 0, 0
    for c, yt, yp in zip(categories, y_true, y_pred):
        if c in hn_cats and yt == 0:
            hn_total += 1
            if yp == 1:
                hn_fp += 1
    hn_fpr = float(hn_fp / max(hn_total, 1))

    report = {
        "dataset": "holdout.csv",
        "total_evaluated": len(df_holdout),
        "calibrated_threshold": threshold,
        "overall": {
            "accuracy": round(acc, 4),
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1": round(f1, 4),
            "fpr": round(fpr, 4),
            "fnr": round(fnr, 4),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
        },
        "per_language": per_lang,
        "per_category": per_cat,
        "hard_negative_fpr": round(hn_fpr, 4),
        "hard_negative_fps": hn_fp,
        "hard_negative_support": hn_total,
        "cpu_latency": {
            "mean_ms": round(float(np.mean(latencies_ms)), 2),
            "p50_ms": round(float(np.percentile(latencies_ms, 50)), 2),
            "p95_ms": round(float(np.percentile(latencies_ms, 95)), 2)
        }
    }

    print("=" * 75)
    print(f"UNBIASED HOLDOUT EVALUATION REPORT ({selected_model_name} @ T* = {threshold})")
    print("=" * 75)
    print(f"Overall Accuracy:       {acc:.4f}")
    print(f"Overall Precision:      {prec:.4f}")
    print(f"Overall Recall:         {rec:.4f}")
    print(f"Overall F1-Score:       {f1:.4f}")
    print(f"False Positive Rate:    {fpr:.4f} ({fp} false alarms / {fp+tn} benign)")
    print(f"False Negative Rate:    {fnr:.4f} ({fn} missed scams / {fn+tp} suspicious)")
    print(f"Hard-Negative FPR:      {hn_fpr:.4f} ({hn_fp} false alarms / {hn_total} legitimate banking/OTPs)")
    print(f"CPU Mean Latency:       {report['cpu_latency']['mean_ms']:.1f} ms / sample (P95: {report['cpu_latency']['p95_ms']:.1f} ms)")
    print("-" * 75)
    print("Language Breakdown:")
    for lang, m in per_lang.items():
        print(f"  - {lang.upper():<10} F1: {m['f1']:.4f} | Precision: {m['precision']:.4f} | Recall: {m['recall']:.4f} | FPR: {m['fpr']:.4f} (n={m['support']})")
    print("=" * 75)

    return report

holdout_report = evaluate_on_untouched_holdout(selected_model_dir, df_holdout, calib_threshold)
with open(ARTIFACTS_DIR / "holdout_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(holdout_report, f, indent=2)

## 11. Dedicated Standalone Export-Only Workflow (Frozen Checkpoint -> ONNX)
This standalone workflow exports the fine-tuned and frozen XLM-RoBERTa checkpoint to an architecture-aware ONNX graph.

### 🛡️ Mandatory Production Constraints Enforced:
1. **Zero Retraining:** Loads fine-tuned weights directly via `from_pretrained()`. Does NOT invoke `Trainer.train()`, optimizer steps, or backward pass.
2. **Zero Recalibration:** Preserves calibrated decision threshold **$T^* = 0.10$** and objective ($\text{Score} = F_1 - 2.0 \cdot \text{FPR}$).
3. **Zero Holdout Exposure:** Holdout dataset (`holdout.csv`) is never loaded, evaluated, or tuned against in this workflow.
4. **Zero Model Switching:** Does NOT train or evaluate Hing-BERT.
5. **Contract Enforcement:** Validates 2 input tensors (`input_ids`, `attention_mask`), no `token_type_ids`, `num_labels = 2`, and canonical label mapping.
6. **Representative Parity Validation:** Verifies numerical agreement ($< 10^{-4}$) and 100% decision match across 10 multilingual representative test samples at $T^* = 0.10$.
7. **Cryptographic Provenance:** Generates `model_metadata.json` and `manifest.json` with SHA-256 digests of all checkpoint, tokenizer, and ONNX assets.
8. **Reproducible Bundle Packaging:** Packages all assets into `satark_xlm_roberta_export_bundle.zip` for one-click local extraction.

In [ ]:
import os
import sys
import io
import json
import time
import shutil
import hashlib
import zipfile
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ARTIFACTS_DIR = Path("ml_service/artifacts")
TOKENIZER_DIR = ARTIFACTS_DIR / "tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
ONNX_FP32_PATH = ARTIFACTS_DIR / "model.onnx"

FROZEN_MODEL_IDENTIFIER = "FacebookAI/xlm-roberta-base"
FROZEN_THRESHOLD = 0.10
FROZEN_OBJECTIVE = "Score = F1 - (2.0 * FPR)"
FROZEN_PENALTY_LAMBDA = 2.0

LABEL2ID = {"BENIGN": 0, "SUSPICIOUS": 1}
ID2LABEL = {0: "BENIGN", 1: "SUSPICIOUS"}

print("=" * 75)
print("SATARK STANDALONE ONNX EXPORT & VERIFICATION WORKFLOW")
print("=" * 75)

# 1. Resolve Checkpoint Directory Location
checkpoint_dir = None
if "selected_model_dir" in locals() and selected_model_dir is not None and Path(selected_model_dir).exists():
    checkpoint_dir = Path(selected_model_dir)
else:
    candidate_paths = [
        ARTIFACTS_DIR / "xlm_roberta" / "final_model",
        ARTIFACTS_DIR / "checkpoint",
        Path("checkpoint"),
    ]
    for p in candidate_paths:
        if p.exists() and (p / "config.json").exists():
            checkpoint_dir = p
            break

if checkpoint_dir is None:
    try:
        from google.colab import files
        print("Checkpoint not found in standard paths.")
        print("Please upload checkpoint archive (.zip) or checkpoint files:")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            if fname.endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(content)) as z:
                    z.extractall(ARTIFACTS_DIR / "checkpoint")
                checkpoint_dir = ARTIFACTS_DIR / "checkpoint"
                break
            else:
                dest = ARTIFACTS_DIR / "checkpoint" / fname
                dest.parent.mkdir(parents=True, exist_ok=True)
                with open(dest, "wb") as f:
                    f.write(content)
                checkpoint_dir = ARTIFACTS_DIR / "checkpoint"
    except Exception as e:
        print(f"Interactive upload skipped: {e}")

assert checkpoint_dir is not None and checkpoint_dir.exists(), (
    f"Checkpoint directory not found! Checked candidate locations: "
    f"{[str(ARTIFACTS_DIR / 'xlm_roberta' / 'final_model'), str(ARTIFACTS_DIR / 'checkpoint')]}"
)
print(f"Checkpoint Directory Located: {checkpoint_dir}")

# 2. Verify Checkpoint Assets
config_file = checkpoint_dir / "config.json"
assert config_file.exists(), f"Missing config.json in {checkpoint_dir}"

weight_files = list(checkpoint_dir.glob("*.safetensors")) + list(checkpoint_dir.glob("*.bin"))
assert len(weight_files) > 0, f"No model weights (*.safetensors or *.bin) found in {checkpoint_dir}"
print(f"Weight Files Verified:         {[w.name for w in weight_files]}")

# 3. Verify Model Configuration & Contract
with open(config_file, "r", encoding="utf-8") as f:
    config_dict = json.load(f)

model_type = config_dict.get("model_type", "")
archs = config_dict.get("architectures", [])
num_labels = config_dict.get("num_labels", 2)
id2label = config_dict.get("id2label", {"0": "BENIGN", "1": "SUSPICIOUS"})

assert "roberta" in model_type.lower() or any("roberta" in a.lower() for a in archs), (
    f"Expected XLM-RoBERTa architecture, got model_type='{model_type}', archs={archs}"
)
assert num_labels == 2, f"Expected 2 labels, got {num_labels}"
print(f"Model Configuration Verified:  Architecture={archs}, Labels={num_labels}, Mapping={id2label}")

# 4. Load Model and Tokenizer Directly (Evaluation Mode - Zero Training)
print(f"Loading PyTorch checkpoint from '{checkpoint_dir}' (zero gradient steps)...")
model = AutoModelForSequenceClassification.from_pretrained(str(checkpoint_dir), local_files_only=True, trust_remote_code=False, use_safetensors=True)
model.eval()

try:
    tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir), local_files_only=True, trust_remote_code=False)
    print(f"Loaded tokenizer from '{checkpoint_dir}'")
except Exception:
    print(f"Tokenizer not found in checkpoint dir; loading from base '{FROZEN_MODEL_IDENTIFIER}'...")
    tokenizer = AutoTokenizer.from_pretrained(FROZEN_MODEL_IDENTIFIER, trust_remote_code=False)

# Save tokenizer assets directly to tokenizer destination directory
tokenizer.save_pretrained(str(TOKENIZER_DIR))
print(f"Tokenizer assets saved to:     {TOKENIZER_DIR}")

# 5. Architecture-Aware Input Contract Inspection & Export to ONNX
sample_text = "Dear user, your bank transaction alert requires immediate verification."
dummy_enc = tokenizer(sample_text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

# XLM-RoBERTa strictly uses input_ids and attention_mask (NO token_type_ids)
input_names = ["input_ids", "attention_mask"]
dummy_input = (dummy_enc["input_ids"], dummy_enc["attention_mask"])
dynamic_axes = {
    "input_ids": {0: "batch_size", 1: "sequence_length"},
    "attention_mask": {0: "batch_size", 1: "sequence_length"},
    "logits": {0: "batch_size"}
}
output_names = ["logits"]

print(f"\nExporting PyTorch model to ONNX: {ONNX_FP32_PATH} (opset=17)...")
torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_FP32_PATH),
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
    opset_version=17,
    dynamo=False  # Preserve the reviewed legacy dynamic_axes export contract.
)

fp32_size_mb = ONNX_FP32_PATH.stat().st_size / (1024 * 1024)
print(f"ONNX Model File Size:          {fp32_size_mb:.2f} MB")

# Validate Graph Structure via onnx.checker
onnx_proto = onnx.load(str(ONNX_FP32_PATH))
onnx.checker.check_model(onnx_proto)
actual_graph_inputs = [inp.name for inp in onnx_proto.graph.input]
print(f"Inspected ONNX Graph Inputs:   {actual_graph_inputs}")
assert actual_graph_inputs == input_names, f"Input signature mismatch: {actual_graph_inputs} vs {input_names}"
print("ONNX Checker & Graph Inputs:   PASSED")

# 6. Representative Parity Benchmark (PyTorch vs ONNX Runtime @ T* = 0.10)
REPRESENTATIVE_PARITY_SAMPLES = [
    {"id": "EN_LEGIT_OTP", "category": "otp_negation", "lang": "en", "text": "Your OTP for HDFC NetBanking login is 849201. Never share this OTP with bank executives, callers, or via SMS.", "expected": "BENIGN"},
    {"id": "EN_LEGIT_TXN", "category": "bank_transactional", "lang": "en", "text": "A/c *1049 debited by Rs 450.00 on 02-Nov-26 via UPI Ref 682910482. Balance Rs 18,420.50.", "expected": "BENIGN"},
    {"id": "EN_LEGIT_UTIL", "category": "utility_service_notice", "lang": "en", "text": "Dear customer, your scheduled fiber maintenance will occur tonight between 2:00 AM and 4:00 AM. Inconvenience regretted.", "expected": "BENIGN"},
    {"id": "HI_LEGIT_OTP", "category": "otp_negation", "lang": "hi", "text": "स्टेट बैंक ऑफ इंडिया: आपका नेट बैंकिंग ओटीपी 391028 है। यह कोड किसी भी व्यक्ति, बैंक कर्मचारी अथवा कॉलर के साथ साझा न करें।", "expected": "BENIGN"},
    {"id": "HINGLISH_LEGIT_TXN", "category": "bank_transactional", "lang": "hinglish", "text": "Aapke savings a/c XX4912 se Rs 1,200 ATM withdrawal successful hua. Current balance check karne ke liye missed call dein: 1800112211.", "expected": "BENIGN"},
    {"id": "EN_SCAM_ELECTRICITY", "category": "electricity_disconnection", "lang": "en", "text": "URGENT NOTICE: Your electricity connection will be disconnected tonight at 9:30 PM due to unpaid bill of Rs 1,480. Call lineman immediately: 9876543210.", "expected": "SUSPICIOUS"},
    {"id": "HI_SCAM_DIGITAL_ARREST", "category": "digital_arrest_impersonation", "lang": "hi", "text": "केंद्रीय जांच ब्यूरो (CBI) दिल्ली: आपके आधार कार्ड का उपयोग अवैध मनी लॉन्ड्रिंग और ड्रग्स पार्सल में पाया गया है। तत्काल डिजिटल अरेस्ट सेशन हेतु स्काइप पर जुड़ें: <url>।", "expected": "SUSPICIOUS"},
    {"id": "HINGLISH_SCAM_JOB", "category": "part_time_job_scam", "lang": "hinglish", "text": "Daily 2-3 hours YouTube videos like karein aur Rs 3,000-8,000 kamayein. No experience required. Telegram par message karein: <url>.", "expected": "SUSPICIOUS"},
    {"id": "HINGLISH_SCAM_LOTTERY", "category": "lottery_advance_fee", "lang": "hinglish", "text": "Congratulations! Aapka mobile number Kaun Banega Crorepati WhatsApp Bumper Draw me Rs 25 Lakhs ka winner bana hai. Claim fee pay karein: <url>.", "expected": "SUSPICIOUS"},
    {"id": "EN_SCAM_APK", "category": "malicious_apk_qr_scam", "lang": "en", "text": "Urgent security update: Your mobile banking app requires immediate upgrade. Download safety patch BharatBank_Assist.apk from: <url>.", "expected": "SUSPICIOUS"}
]

ort_session = ort.InferenceSession(str(ONNX_FP32_PATH), providers=["CPUExecutionProvider"])
max_abs_diff = 0.0
max_rel_diff = 0.0
parity_results = []

print("\n" + "=" * 75)
print(f"REPRESENTATIVE PARITY BENCHMARK (PyTorch vs ONNX Runtime @ T* = {FROZEN_THRESHOLD})")
print("=" * 75)

for s in REPRESENTATIVE_PARITY_SAMPLES:
    enc = tokenizer(s["text"], max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    pt_feed = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
    with torch.no_grad():
        pt_out = model(**pt_feed).logits.detach().cpu().numpy()[0]

    ort_feed = {
        "input_ids": enc["input_ids"].numpy().astype(np.int64),
        "attention_mask": enc["attention_mask"].numpy().astype(np.int64)
    }
    ort_out = ort_session.run(output_names, ort_feed)[0][0]

    pt_exp = np.exp(pt_out - np.max(pt_out))
    pt_prob = float((pt_exp / np.sum(pt_exp))[1])
    ort_exp = np.exp(ort_out - np.max(ort_out))
    ort_prob = float((ort_exp / np.sum(ort_exp))[1])

    pt_decision = "SUSPICIOUS" if pt_prob >= FROZEN_THRESHOLD else "BENIGN"
    ort_decision = "SUSPICIOUS" if ort_prob >= FROZEN_THRESHOLD else "BENIGN"

    abs_d = float(np.max(np.abs(pt_out - ort_out)))
    rel_d = float(np.max(np.abs(pt_out - ort_out) / (np.abs(pt_out) + 1e-7)))
    if abs_d > max_abs_diff:
        max_abs_diff = abs_d
    if rel_d > max_rel_diff:
        max_rel_diff = rel_d

    parity_results.append({
        "sample_id": s["id"],
        "lang": s["lang"],
        "pt_prob": round(pt_prob, 5),
        "ort_prob": round(ort_prob, 5),
        "pt_decision": pt_decision,
        "ort_decision": ort_decision,
        "abs_diff": abs_d,
        "match": (pt_decision == ort_decision) and (abs_d < 1e-4)
    })
    print(f"  {s['id']:<25} Diff: {abs_d:.6e} | PT: {pt_prob:.4f} ({pt_decision}) | ORT: {ort_prob:.4f} ({ort_decision})")

print("-" * 75)
print(f"Max Absolute Discrepancy:     {max_abs_diff:.6e} (Tolerance Bound: 1.000000e-04)")
print(f"Max Relative Discrepancy:     {max_rel_diff:.6e}")
assert max_abs_diff < 1e-4, f"Parity failure: max abs discrepancy {max_abs_diff} exceeds 1e-4"
assert all(r["match"] for r in parity_results), "Decision mismatch between PyTorch and ONNX Runtime!"
print("Numerical Parity Benchmark:   PASSED (100% decision match, all discrepancies < 1e-4)")
print("=" * 75)

# 7. Write Model Metadata (Preserving T* = 0.10)
meta = {
    "model_name": FROZEN_MODEL_IDENTIFIER,
    "architecture": "XLMRobertaForSequenceClassification",
    "base_model": FROZEN_MODEL_IDENTIFIER,
    "tokenizer_class": tokenizer.__class__.__name__,
    "num_labels": 2,
    "id2label": ID2LABEL,
    "label2id": LABEL2ID,
    "input_names": input_names,
    "output_names": output_names,
    "uses_token_type_ids": False,
    "opset_version": 17,
    "max_length": 128,
    "optimal_threshold": FROZEN_THRESHOLD,
    "threshold_objective": FROZEN_OBJECTIVE,
    "threshold_penalty_lambda": FROZEN_PENALTY_LAMBDA,
    "actual_file_size_mb": round(fp32_size_mb, 2),
    "max_abs_parity_discrepancy": max_abs_diff,
    "max_rel_parity_discrepancy": max_rel_diff,
    "validation_status": "PASSED",
    "export_contract_version": "2.0-phase6"
}
with open(ARTIFACTS_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("Model metadata written to:     ml_service/artifacts/model_metadata.json")

# 8. Checksum Calculation & Reproducibility Manifest
def sha256_file(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

checksums = {}
checksums["model.onnx"] = sha256_file(ONNX_FP32_PATH)
checksums["model_metadata.json"] = sha256_file(ARTIFACTS_DIR / "model_metadata.json")

for tok_f in TOKENIZER_DIR.glob("*"):
    if tok_f.is_file():
        checksums[f"tokenizer/{tok_f.name}"] = sha256_file(tok_f)

for cp_f in checkpoint_dir.glob("*"):
    if cp_f.is_file():
        checksums[f"checkpoint/{cp_f.name}"] = sha256_file(cp_f)

manifest = {
    "manifest_version": "1.0",
    "project": "SATARK",
    "frozen_model_contract": {
        "model_identifier": FROZEN_MODEL_IDENTIFIER,
        "architecture": "XLMRobertaForSequenceClassification",
        "base_model": FROZEN_MODEL_IDENTIFIER,
        "tokenizer_class": tokenizer.__class__.__name__,
        "positive_class": "SUSPICIOUS",
        "positive_class_index": 1,
        "label_mapping": {"id2label": ID2LABEL, "label2id": LABEL2ID},
        "calibrated_threshold": FROZEN_THRESHOLD,
        "threshold_objective": FROZEN_OBJECTIVE,
        "threshold_penalty_lambda": FROZEN_PENALTY_LAMBDA
    },
    "export_configuration": {
        "opset_version": 17,
        "max_length": 128,
        "input_names": input_names,
        "output_names": output_names,
        "dynamic_axes": dynamic_axes,
        "uses_token_type_ids": False,
        "constant_folding": True
    },
    "runtime_environment_contract": {
        "python_version": sys.version.split()[0],
        "torch_version": torch.__version__,
        "transformers_version": getattr(sys.modules.get("transformers"), "__version__", "unknown"),
        "onnx_version": onnx.__version__,
        "onnxruntime_version": ort.__version__
    },
    "dataset_partition_hashes_sha256": {
        "train_csv": "ca81bea012ca78689ad0a4b3df4cb6cdcfc3223893af8a68e87990b4fe717bb9",
        "calibration_csv": "1ecaa64a6851d3c6b544215aa8d20390c65d1e796166880f7c3c2c48d51ed784",
        "holdout_csv": "8d83ee0d3d7c7062687c8bc867fed0bb0d9b870c1a235758b2c8c6e94b1ba58f",
        "training_corpus_4500_csv": "943ed7e4d2c41f90e49865a6374214ac9782ec4d904d5fdc490d0c006719a0d4"
    },
    "frozen_evaluation_fixtures_sha256": {
        "eval_test_set_csv": "09661d914ddd90ca1cf2de78abd2db49e438b20638c4fd71e4e01607c6b9d86f"
    },
    "checksums_sha256": checksums,
    "parity_summary": {
        "samples_evaluated": len(REPRESENTATIVE_PARITY_SAMPLES),
        "max_abs_discrepancy": max_abs_diff,
        "max_rel_discrepancy": max_rel_diff,
        "tolerance_abs_bound": 1e-4,
        "tolerance_rel_bound": 1e-3,
        "all_decisions_matched": all(r["match"] for r in parity_results),
        "status": "PASSED"
    },
    "checkpoint_freeze_rule": "Original checkpoint files must never be overwritten",
    "export_timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}

manifest_path = ARTIFACTS_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Reproducibility manifest:      ml_service/artifacts/manifest.json")

# 9. Packaging Reproducible Bundle
BUNDLE_ZIP = Path("satark_xlm_roberta_export_bundle.zip")
with zipfile.ZipFile(BUNDLE_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_FP32_PATH, arcname="model.onnx")
    z.write(ARTIFACTS_DIR / "model_metadata.json", arcname="model_metadata.json")
    z.write(manifest_path, arcname="manifest.json")
    for tok_f in TOKENIZER_DIR.glob("*"):
        if tok_f.is_file():
            z.write(tok_f, arcname=f"tokenizer/{tok_f.name}")
    for cp_f in checkpoint_dir.glob("*"):
        if cp_f.is_file():
            z.write(cp_f, arcname=f"checkpoint/{cp_f.name}")

# Also create backward-compatible alias
shutil.copyfile(BUNDLE_ZIP, "satark_ml_artifacts.zip")

bundle_size = round(BUNDLE_ZIP.stat().st_size / (1024 * 1024), 2)
print(f"\nBundle Created Successfully:   {BUNDLE_ZIP} ({bundle_size} MB)")
print("Archived Contents: model.onnx, model_metadata.json, manifest.json, tokenizer/, checkpoint/")

try:
    from google.colab import files
    files.download(str(BUNDLE_ZIP))
    print("Automatic browser download triggered in Google Colab.")
except ImportError:
    print(f"Local runtime detected: bundle available at {BUNDLE_ZIP.resolve()}")

## 12. Optional INT8 Dynamic Quantization & CPU Benchmark
Apply INT8 dynamic quantization to the FP32 ONNX graph.

Verification Standard:
1. Verify INT8 numerical agreement and class consistency against PyTorch.
2. Measure **actual** file size reduction and **actual** latency speedup on CPU (no assumed compression or acceleration).

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_INT8_PATH = ARTIFACTS_DIR / "model_quantized.onnx"

print("Applying INT8 dynamic quantization...")
quantize_dynamic(
    model_input=str(ONNX_FP32_PATH),
    model_output=str(ONNX_INT8_PATH),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul", "Gemm", "Gather"]
)

int8_size_mb = ONNX_INT8_PATH.stat().st_size / (1024 * 1024)
actual_reduction_pct = (1.0 - (int8_size_mb / fp32_size_mb)) * 100.0
print(f"Quantized Model Generated: {int8_size_mb:.2f} MB (Measured Reduction: {actual_reduction_pct:.1f}%)")

# Benchmark actual CPU latency (FP32 vs INT8)
sess_fp32 = ort.InferenceSession(str(ONNX_FP32_PATH), providers=["CPUExecutionProvider"])
sess_int8 = ort.InferenceSession(str(ONNX_INT8_PATH), providers=["CPUExecutionProvider"])

dummy_feed = {k: np.ones((1, 128), dtype=np.int64) for k in input_names}

# Warmup iterations
for _ in range(5):
    sess_fp32.run(None, dummy_feed)
    sess_int8.run(None, dummy_feed)

# Benchmark FP32
N = 30
t0 = time.perf_counter()
for _ in range(N):
    sess_fp32.run(None, dummy_feed)
fp32_lat = ((time.perf_counter() - t0) / N) * 1000.0

# Benchmark INT8
t0 = time.perf_counter()
for _ in range(N):
    sess_int8.run(None, dummy_feed)
int8_lat = ((time.perf_counter() - t0) / N) * 1000.0

actual_speedup = fp32_lat / max(int8_lat, 0.001)

# Numerical Class Agreement Test on test batch
sample_batch_texts = [
    "Dear customer, your electricity bill of Rs 1450 is unpaid. Power will disconnect tonight.",
    "Your OTP for SBI card login is 482910. Do NOT share this OTP with anyone.",
    "Congratulations! You won Rs 25,00,000 in KBC Lucky Draw. Pay registration fee to claim.",
    "Your package has been delivered to your front door. Thank you for shopping with us."
]
agreements = 0
for text in sample_batch_texts:
    enc = tokenizer(text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    feed = {k: enc[k].numpy() for k in input_names}
    logits_fp32 = sess_fp32.run(None, feed)[0]
    logits_int8 = sess_int8.run(None, feed)[0]
    pred_fp32 = int(np.argmax(logits_fp32))
    pred_int8 = int(np.argmax(logits_int8))
    if pred_fp32 == pred_int8:
        agreements += 1

agreement_rate = float(agreements / len(sample_batch_texts))
print(f"INT8 / FP32 Class Agreement on Verification Samples: {agreement_rate * 100:.1f}%")

print("=" * 70)
print("ONNX QUANTIZATION & CPU BENCHMARK REPORT (ACTUAL MEASURED VALUES)")
print("=" * 70)
print(f"FP32 Model Size:     {fp32_size_mb:.2f} MB")
print(f"INT8 Model Size:     {int8_size_mb:.2f} MB ({actual_reduction_pct:.1f}% measured reduction)")
print(f"FP32 Mean Latency:   {fp32_lat:.2f} ms / sample")
print(f"INT8 Mean Latency:   {int8_lat:.2f} ms / sample ({actual_speedup:.2f}x measured speedup)")
print(f"Class Agreement:     {agreement_rate * 100:.1f}%")
print("=" * 70)

quant_report = {
    "fp32_size_mb": round(fp32_size_mb, 2),
    "int8_size_mb": round(int8_size_mb, 2),
    "measured_size_reduction_pct": round(actual_reduction_pct, 2),
    "fp32_latency_ms": round(fp32_lat, 2),
    "int8_latency_ms": round(int8_lat, 2),
    "measured_speedup": round(actual_speedup, 2),
    "class_agreement_rate": round(agreement_rate, 4)
}
with open(ARTIFACTS_DIR / "quantization_report.json", "w", encoding="utf-8") as f:
    json.dump(quant_report, f, indent=2)

# Append INT8 model and report to existing bundle zip if created
for zip_candidate in [Path("satark_xlm_roberta_export_bundle.zip"), Path("satark_ml_artifacts.zip")]:
    if zip_candidate.exists():
        with zipfile.ZipFile(zip_candidate, "a", zipfile.ZIP_DEFLATED) as z:
            z.write(ONNX_INT8_PATH, arcname="model_quantized.onnx")
            z.write(ARTIFACTS_DIR / "quantization_report.json", arcname="quantization_report.json")
        print(f"Appended INT8 model and report to: {zip_candidate}")

## 13. Local SATARK Deployment & Governance Standard
Once `satark_xlm_roberta_export_bundle.zip` (or `satark_ml_artifacts.zip`) is downloaded:

### 1. Extract Artifacts into `ml_service/artifacts/`
Extract the zip directly into your local repository root:
`c:\Users\LENOVO\Downloads\Satark 2.0\ml_service\artifacts\`

Expected Directory Structure after extraction:
```
ml_service/artifacts/
├── model.onnx                     (FP32 base ONNX model)
├── model_metadata.json            (Model signatures, calibrated threshold T* = 0.10)
├── manifest.json                  (Cryptographic checksums, dataset hashes, versions)
├── tokenizer/                     (Local SentencePiece & Fast tokenizer assets)
│   ├── sentencepiece.bpe.model
│   ├── tokenizer.json
│   ├── tokenizer_config.json
│   └── special_tokens_map.json
├── checkpoint/                    (Preserved PyTorch weights & configuration)
│   ├── model.safetensors
│   └── config.json
├── model_quantized.onnx           (Optional INT8 dynamically quantized model)
└── quantization_report.json       (Optional measured compression and latency profile)
```

### 2. Verify Local ONNX Inference & Parity
Run the local CLI evaluation against the untouched holdout partition:
```powershell
$env:PYTHONPATH = "."
python -m ml_service.training.evaluate --model_path ml_service/artifacts/model.onnx --data_file ml_service/dataset/holdout.csv
```

### 3. Run Repository Automated Test Suite
```powershell
$env:PYTHONPATH = "."
python -m unittest discover -s ml_service/tests -p "test_*.py"
```

---

### 🛡️ Critical Standing Governance Rules:
- **Rule Engine Sovereignty:** The deterministic Rule Engine remains the authoritative first line of defense. The ML model acts strictly as supplementary evidence; it never overrides deterministic rule verdicts.
- **Frozen Evaluator Preservation:** `eval/test_set.csv` must remain frozen and unmodified. Never evaluate training or threshold tuning against evaluation fixtures.
- **`ML_ENABLED=false` Standard:** ML inference remains disabled by feature flag until local operational verification is complete.